# Los números cuánticos en términos qg, y Λ como filtro en LiH (§155, §153)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Viny2030/qang/blob/main/notebooks/qang_quantum_numbers_es.ipynb)

**Parte A (§155)** lee cada número cuántico de un electrón con qang: m_s del qg_Z de un qubit, m_l del peso de Hamming de un registro simétrico (así el filtro de peso qg es la proyección sobre un valor de m_l), l del valor qang de SWAP, y la capa n del hidrógeno como dos espines (n−1)/2. Identidades exactas, mostradas con circuitos muestreados.

**Parte B (§153)** usa el m_l molecular, Λ, como tercer filtro en el estado fundamental de LiH con 10 qubits, junto a los filtros de número de electrones, con el modelo de ruido de FakeTorino.

Todo corre en simuladores locales (sin cuenta, sin hardware), en unos 15 minutos (casi todo en la parte de LiH, Parte B).

## 1. Instalación

In [ ]:
import importlib, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "qang", "qiskit-aer", "qiskit-ibm-runtime", "scipy"], check=False)
importlib.invalidate_caches()
try:
    from qang import angular, ibm
    ok = hasattr(angular, "j_squared_from_swaps") and hasattr(ibm, "best_chain")
except ImportError:
    ok = False
if not ok:  # the published qang predates §155: install from GitHub
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
                    "git+https://github.com/Viny2030/qang.git"], check=True)
    print("installed qang from GitHub; if an import fails below, restart the runtime once")
import qang
print("qang", qang.__version__)

## 2. Configuración común

`orbital_circuit` prepara los orbitales l = 1 y el singlete l = 0 en dos qubits (m_l = 1 − peso de Hamming); `with_measure(..., bell=True)` rota a la base de Bell para que el singlete sea el resultado 11, y así se lee qg_SWAP = ⟨SWAP⟩ = 1 − 2 P(singlete). Copiado de `examples/quantum_numbers_qg.py`.

In [ ]:
import math, time
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qang import angular as A
from qang import interop as IO
from qang import ibm
from qang.core import Qang

SHOTS = 20000
T0 = time.time()


def orbital_circuit(name):
    qc = QuantumCircuit(2)
    if name == "p-1":
        qc.x([0, 1])
    elif name in ("p0", "pz"):  # (|01> + |10>)/sqrt2
        qc.h(0); qc.cx(0, 1); qc.x(1)
    elif name == "px":  # proportional to |00> - |11>
        qc.h(0); qc.z(0); qc.cx(0, 1)
    elif name == "py":  # proportional to |00> + |11>
        qc.h(0); qc.cx(0, 1)
    elif name == "s":  # singlet (|01> - |10>)/sqrt2, l = 0
        qc.h(0); qc.z(0); qc.cx(0, 1); qc.x(1)
    return qc


def with_measure(qc, bell=False):
    c = qc.copy()
    if bell:
        c.cx(0, 1)
        c.h(0)
    c.measure_all()
    return c


def sample(circs, backend=None, seed=7):
    sim = AerSimulator.from_backend(backend, seed_simulator=seed) if backend else AerSimulator(seed_simulator=seed)
    t = transpile(circs, sim, optimization_level=1, seed_transpiler=137)
    res = sim.run(t, shots=SHOTS).result()
    return [IO.canonical_counts(res.get_counts(i), sdk="qiskit") for i in range(len(circs))]


def m_distribution(counts, n):
    pm = np.zeros(n + 1)
    shots = sum(counts.values())
    for k, c in counts.items():
        pm[sum(k)] += c / shots
    return pm


def qg_swap(bell_counts):
    shots = sum(bell_counts.values())
    return 1 - 2 * bell_counts.get((1, 1), 0) / shots

## 3. m_s: un qubit (§155)

Para un espín ½, qg_Z = 2 m_s y el ángulo qang es la inclinación del espín. Publicado: RY(60°) da qg_Z = +0.502 y el ángulo 59.9°.

In [ ]:
for theta in (0.0, math.pi / 3, math.pi / 2, math.pi):
    qc = QuantumCircuit(1)
    qc.ry(theta, 0)
    (counts,) = sample([with_measure(qc)])
    z, se = IO.qg_z_profile(counts)[0]
    q = Qang(z)
    print(f"RY({math.degrees(theta):5.1f} deg): qg_Z {q.value:+.3f} = {q.milliqang:+7.1f} mqg, "
          f"qang angle {q.to_theta('deg'):5.1f} deg, <m_s> = {z / 2:+.3f}, qg_S = {q.to_entropic().value:.3f}")

## 4. m_l y l: un registro de dos qubits (§155)

Un orbital l = 1 es un espín 1 = dos qubits en su subespacio simétrico: m_l = 1 − peso de Hamming, y el qg_Z del registro (promedio sobre sus qubits) es ⟨m_l⟩/l. Los p_x y p_y reales son superposiciones de m_l = ±1: su qg_Z es 0 como el de p_z, pero su distribución de m_l está repartida (qg_S(m) = log 2 / log 3 = 0.63); por eso §153 necesita los orbitales complejos π±. l sale de SWAP: ⟨J²⟩ = 1 + qg_SWAP, así que qg_SWAP = +1 significa l = 1 y −1 significa l = 0 (el singlete).

In [ ]:
names = ("p+1", "p0", "p-1", "px", "py", "pz", "s")
zc = sample([with_measure(orbital_circuit(n)) for n in names])
bc = sample([with_measure(orbital_circuit(n), bell=True) for n in names])
print("orbital  P(m_l=+1,0,-1)        register qg_Z  qg_S(m)  qg_SWAP  l")
for name, cz, cb in zip(names, zc, bc):
    pm = m_distribution(cz, 2)  # index = weight: m_l = +1, 0, -1
    reg = float(np.mean([z for z, _ in IO.qg_z_profile(cz)]))
    sw = qg_swap(cb)
    l_hat = A.l_from_j_squared(A.j_squared_from_swaps(2, [sw]))
    print(f"{name:7s}  {np.round(pm, 3)!s:22s} {reg:+.3f}         {A.qg_s_spin(pm):.3f}    {sw:+.3f}   {l_hat:.2f}")

## 5. La capa n (§155)

La simetría oculta SO(4) del hidrógeno hace de la capa n dos espines (n−1)/2: dos registros simétricos de n−1 qubits cada uno. Para n = 2 es el registro de dos qubits de arriba: (qg_SWAP, qg_Z del registro) da 2s y los tres 2p. Para n = 3 (estados exactos en 4 qubits) el qg_Z del registro es m_l/2, así que los ángulos qang son 0°, 60°, 90°, 120°, 180°.

El ángulo qang de un autoestado |l, m⟩ es arccos(m/l); el ángulo del modelo vectorial de los libros, arccos(m/√(l(l+1))), queda siempre más cerca del ecuador, porque el estado tiene una dispersión transversal l(l+1) − m².

In [ ]:
labels = {"2s": "s", "2p+1": "p+1", "2p0": "p0", "2p-1": "p-1"}
zc = sample([with_measure(orbital_circuit(n)) for n in labels.values()])
bc = sample([with_measure(orbital_circuit(n), bell=True) for n in labels.values()])
print("n = 2:")
for label, cz, cb in zip(labels, zc, bc):
    sw = qg_swap(cb)
    reg = float(np.mean([z for z, _ in IO.qg_z_profile(cz)]))
    l_hat = round(A.l_from_j_squared(A.j_squared_from_swaps(2, [sw])))
    print(f"  {label:5s} qg_SWAP {sw:+.3f}, register qg_Z {reg:+.3f}  ->  l = {l_hat}, m_l = {round(reg):+d}")

print("n = 3 (two symmetric 2-qubit registers, 9 states):")
k = 2
sym = np.array([A.dicke_state(k, w) for w in range(k + 1)]).T
P = np.kron(sym, sym)
Jx, Jy, Jz = A.spin_operators(2 * k)
J2op = Jx @ Jx + Jy @ Jy + Jz @ Jz
ev, vec = np.linalg.eigh(P.conj().T @ J2op @ P + 1e-3 * P.conj().T @ Jz @ P)  # split the m inside each l
rows = []
for v in vec.T:
    psi = P @ v
    l = (-1 + math.sqrt(1 + 4 * np.real(np.vdot(psi, J2op @ psi)))) / 2
    z = np.real(np.vdot(psi, Jz @ psi)) / k
    rows.append((round(l), round(z * k), z))
for l, m, z in sorted(rows, key=lambda r: (r[0], -r[1])):
    print(f"  l = {l}, m_l = {m:+d}: register qg_Z {z:+.3f}, qang angle {Qang(z).to_theta('deg'):5.1f} deg")

print("qang angle vs vector-model angle:")
for l, m in ((1, 1), (2, 2), (2, 1), (3, 3), (3, 1)):
    print(f"  l = {l}, m = {m:+d}: qang {math.degrees(A.qang_angle_of_eigenstate(l, m)):5.1f} deg, "
          f"vector model {math.degrees(A.vector_model_angle(l, m)):5.1f} deg, <Jx^2+Jy^2> = {A.transverse_spread(l, m):.0f}")

## 6. Con ruido: el filtro de peso es la proyección m_l = 0 (§155)

p_z tiene m_l = 0 exactamente. Con el modelo de ruido de FakeTorino se escapa a m_l = ±1: en el 19% de los disparos con el layout por defecto (cuyo qubit 0 tiene 17% de error de lectura) y en un 2% con la cadena elegida por `ibm.best_chain`, que además predice la fracción conservada. En ambos casos el filtro de peso devuelve P(m_l = 0) = 1.

In [ ]:
import qiskit_ibm_runtime.fake_provider as fp

b = fp.FakeTorino()
sim = AerSimulator.from_backend(b, seed_simulator=7)
qc = with_measure(orbital_circuit("pz"))
default = transpile(qc, b, optimization_level=1, seed_transpiler=137)
layout, pred, chosen = ibm.best_chain(qc, b, weight=1)
for label, t in (("default layout", default), ("ibm.best_chain", chosen)):
    phys = ibm.measured_physical_qubits(t)
    cz = IO.canonical_counts(sim.run(t, shots=SHOTS).result().get_counts(), sdk="qiskit")
    pm = m_distribution(cz, 2)
    kept, frac = IO.weight_filter(cz, 1)
    pmf = m_distribution(kept, 2)
    print(f"{label} {phys}: readout errors {[round(max(e), 3) for e in ibm.readout_errors(b, phys)]}")
    print(f"  raw      P(m_l=+1,0,-1) = {np.round(pm, 4)}")
    print(f"  filtered P(m_l=+1,0,-1) = {np.round(pmf, 4)}, kept {frac:.3f}"
          + (f" (predicted {pred['kept']:.3f})" if t is chosen else ""))

## 7. Λ como tercer filtro en LiH (§153)

Una molécula lineal conserva Λ = Σ m_l(i) n_i. Escribiendo los orbitales π como π± (m_l = ±1) en lugar de p_x, p_y, Λ es diagonal y se puede chequear disparo por disparo, como los números de electrones N↑, N↓; y solo necesita los cuatro qubits π en la base Z, así que también alcanza grupos de medición no diagonales. LiH, 1.6 Å, STO-3G, 1s congelado, 10 qubits; el circuito es un cargador unario sobre los cinco pares de electrones (θ publicado, 5.0 mHa sobre CASCI). El Hamiltoniano (PySCF, `tools/make_lih_lambda_hamiltonian.py`) se lee del repositorio.

Publicado en FakeTorino (20 000 disparos por grupo, 317 grupos): error de energía sin filtro 163.8 mHa, filtro N 11.9, solo Λ 147.4, ambos 11.7; ocupación π sin filtro 0.138, con ambos 0.0023 (ideal 0.002). Donde N ya se aplica, Λ casi no agrega nada (un error que conserva N↑, N↓ pero cambia Λ requiere dos errores); es el chequeo correcto para la ocupación π. Acá: 4 000 disparos por grupo.

In [ ]:
import json, os, urllib.request
from qiskit.circuit.library import XXPlusYYGate
from qiskit.quantum_info import SparsePauliOp, Statevector


def repo_json(name):
    local = os.path.join("..", "examples", "data", name)
    if os.path.exists(local):
        return json.load(open(local, encoding="utf-8"))
    url = f"https://raw.githubusercontent.com/Viny2030/qang/main/examples/data/{name}"
    return json.loads(urllib.request.urlopen(url).read().decode("utf-8"))


D = repo_json("lih_lambda_jw.json")
theta = repo_json("lih_lambda_filter_qg.json")["theta"]
NQ, PI_QUBITS = 5, (3, 4, 8, 9)  # qubit p = orbital p spin up, p + 5 spin down; orbitals 3, 4 = pi+, pi-
PARTNER = {0: 5, 1: 6, 2: 7, 3: 9, 4: 8}
KINDS = ("raw", "NS", "L", "NSL")
H = SparsePauliOp.from_list(list(D["terms"].items()))


def ansatz(th, basis=None):
    qc = QuantumCircuit(2 * NQ)
    qc.x(0)
    for k, t in enumerate(th):
        qc.append(XXPlusYYGate(2 * float(t)), [k, k + 1])
    for q, p in PARTNER.items():
        qc.cx(q, p)
    if basis is not None:
        for q, bb in enumerate(reversed(basis)):
            if bb == "X":
                qc.h(q)
            elif bb == "Y":
                qc.sdg(q); qc.h(q)
        qc.measure_all()
    return qc


def groups(H):
    diag, rest = [], []
    for lab, c in zip(H.paulis.to_labels(), H.coeffs):
        (diag if set(lab) <= set("IZ") else rest).append((lab, float(np.real(c))))
    pi_z = lambda lab: all(lab[2 * NQ - 1 - q] in "IZ" for q in PI_QUBITS)
    out = [diag]
    for part in ([t for t in rest if pi_z(t[0])], [t for t in rest if not pi_z(t[0])]):
        for g in SparsePauliOp.from_list(part).group_commuting(qubit_wise=True):
            out.append([(l, float(np.real(c))) for l, c in zip(g.paulis.to_labels(), g.coeffs)])
    return out


def basis_of(group):
    bs = ["Z"] * (2 * NQ)
    for lab, _ in group:
        for i, ch in enumerate(lab):
            if ch != "I":
                bs[i] = ch
    return "".join(bs)


def checks(basis):
    z = [basis[2 * NQ - 1 - q] == "Z" for q in range(2 * NQ)]
    return {"NS": all(z), "L": all(z[q] for q in PI_QUBITS)}


def keep(bit, kind, can):
    if kind in ("NS", "NSL") and can["NS"] and (sum(bit[:NQ]) != 1 or sum(bit[NQ:]) != 1):
        return False
    if kind in ("L", "NSL") and can["L"] and bit[3] + bit[8] - bit[4] - bit[9] != 0:
        return False
    return True


def estimate(counts, group, basis, kind):
    can = checks(basis)
    tot, acc = 0, np.zeros(len(group))
    for s, c in counts.items():
        bit = [int(s[2 * NQ - 1 - q]) for q in range(2 * NQ)]
        if not keep(bit, kind, can):
            continue
        tot += c
        for k, (lab, _) in enumerate(group):
            acc[k] += c * (1 - 2 * (sum(bit[q] for q in range(2 * NQ) if lab[2 * NQ - 1 - q] != "I") % 2))
    return float(sum(cf * v for (_, cf), v in zip(group, acc / max(tot, 1)))), tot


def pi_occupation(counts, kind):
    can = checks("Z" * (2 * NQ))
    tot = occ = 0
    for s, c in counts.items():
        bit = [int(s[2 * NQ - 1 - q]) for q in range(2 * NQ)]
        if keep(bit, kind, can):
            tot += c
            occ += c * sum(bit[q] for q in PI_QUBITS)
    return occ / max(tot, 1)


G = groups(H)
sv = Statevector(ansatz(theta))
exact = [float(np.real(sv.expectation_value(SparsePauliOp.from_list(g)))) for g in G]
print(f"{len(G)} measurement groups; N filter applies to {sum(checks(basis_of(g))['NS'] for g in G)}, "
      f"Lambda to {sum(checks(basis_of(g))['L'] for g in G)}; circuit energy {sum(exact):.5f} Ha "
      f"(CASCI {D['fci_active']:.5f})")

In [ ]:
b = fp.FakeTorino()
bases = [basis_of(g) for g in G]
ts = transpile([ansatz(theta, bs) for bs in bases], b, optimization_level=1, seed_transpiler=137)
try:
    from qiskit_ibm_runtime.executor_sampler import Sampler
except ImportError:
    from qiskit_ibm_runtime import SamplerV2 as Sampler
res = Sampler(mode=b).run(ts, shots=4000).result()
counts = [r.data.meas.get_counts() for r in res]
print("filter   energy error (mHa)   pi occupation (ideal %.4f)" % sum(
    p * sum(int(s[2 * NQ - 1 - q]) for q in PI_QUBITS) for s, p in sv.probabilities_dict().items()))
for kind, label in zip(KINDS, ("none", "N", "Lambda", "N + Lambda")):
    e = sum(estimate(c, g, bs, kind)[0] for c, g, bs in zip(counts, G, bases))
    print(f"{label:11s}  {1000 * abs(e - sum(exact)):8.1f}             {pi_occupation(counts[0], kind):.4f}")
print(f"total time {time.time() - T0:.0f} s")

## Resumen

* **Cada número cuántico tiene una lectura qang.** m_s es el qg_Z de un qubit; m_l es el peso de Hamming de un registro simétrico, así que el filtro de peso qg es la proyección sobre un valor de m_l; l se lee de qg_SWAP; n es el tamaño de dos registros. El ángulo qang de |l, m⟩ es arccos(m/l), no el ángulo del modelo vectorial. Es una traducción, no física nueva.
* **Λ en LiH**: una cantidad conservada se vuelve filtro cuando se eligen los orbitales para que sea diagonal (π± complejos). Agrega poco donde ya se aplica el filtro de número de electrones, y es el chequeo correcto para la ocupación π.

Detalles: `RESEARCH_NOTES.md` §153 y §155, `examples/quantum_numbers_qg.py`, `examples/lih_lambda_filter_qg.py`.